# Module 08-1 솔루션: Exponential Backoff + Jitter 구현

**참조 문서**: `docs/part4-production/08-error-handling-resilience.md` 섹션 2.1

In [ ]:
import sys
sys.path.insert(0, '../..')

import time
import random
import logging
from common.fake_llm import FakeLLM

logging.basicConfig(level=logging.WARNING)
print("환경 설정 완료")

In [ ]:
# 실습 1 솔루션: retry_with_backoff() 함수 구현

def retry_with_backoff(
    fn,
    max_retries: int = 3,
    base_delay: float = 1.0,
    max_delay: float = 30.0,
    backoff_factor: float = 2.0,
    jitter: bool = True,
    retriable_exceptions: tuple = (Exception,),
):
    """Exponential Backoff + Jitter로 함수를 재시도합니다."""
    last_exception = None

    for attempt in range(max_retries + 1):  # 최초 시도 + 재시도
        try:
            return fn()
        except retriable_exceptions as exc:
            last_exception = exc

            if attempt >= max_retries:
                # 마지막 시도도 실패 → 예외 재발생
                raise

            # 대기 시간 계산: base_delay * (backoff_factor ^ attempt)
            delay = min(base_delay * (backoff_factor ** attempt), max_delay)

            # Jitter 추가: 0 ~ delay 사이의 랜덤 값
            if jitter:
                delay = random.uniform(0, delay)

            print(f"재시도 {attempt + 1}/{max_retries}: {delay:.2f}초 대기 (에러: {type(exc).__name__})")
            time.sleep(delay)

    raise last_exception


print("retry_with_backoff 함수 정의 완료")

In [ ]:
# 실습 1 검증: 성공 케이스
call_count = 0

def succeed_on_third():
    global call_count
    call_count += 1
    if call_count < 3:
        raise ConnectionError(f"연결 실패 (시도 {call_count}회)")
    return "성공!"

call_count = 0
result = retry_with_backoff(
    fn=succeed_on_third,
    max_retries=3,
    base_delay=0.01,
    jitter=False,
    retriable_exceptions=(ConnectionError,),
)

assert result == "성공!"
assert call_count == 3
print(f"성공 케이스 통과: {call_count}번 시도 후 성공")

In [ ]:
# 실습 1 검증: 실패 케이스
fail_count = 0

def always_fail():
    global fail_count
    fail_count += 1
    raise ValueError("항상 실패")

fail_count = 0
try:
    retry_with_backoff(fn=always_fail, max_retries=2, base_delay=0.01, jitter=False)
    assert False, "예외가 발생해야 합니다"
except ValueError:
    pass

assert fail_count == 3
print(f"실패 케이스 통과: {fail_count}번 시도 후 예외 발생")
print("실습 1 완료!")

In [ ]:
# 실습 2 솔루션: max_delay 제한 + Jitter 동작 확인
from unittest.mock import patch

sleep_durations = []
MAX_DELAY = 3.0

with patch('time.sleep', side_effect=lambda d: sleep_durations.append(d)):
    try:
        retry_with_backoff(
            fn=lambda: (_ for _ in ()).throw(RuntimeError("fail")),
            max_retries=4,
            base_delay=1.0,
            max_delay=MAX_DELAY,
            backoff_factor=2.0,
            jitter=False,
        )
    except RuntimeError:
        pass

print(f"sleep 호출 대기 시간들: {[round(d, 2) for d in sleep_durations]}")

assert len(sleep_durations) == 4
assert all(d <= MAX_DELAY for d in sleep_durations)
print(f"max_delay 제한 통과: 모든 대기 시간 <= {MAX_DELAY}초")
print("실습 2 완료!")

In [ ]:
# 실습 3 솔루션: FakeLLM과 연동하여 재시도 흐름 시뮬레이션
attempt_count = 0
llm = FakeLLM(responses={"분석": "분석 완료: 버그 없음"})

def flaky_llm_call():
    global attempt_count
    attempt_count += 1
    if attempt_count <= 2:
        raise ConnectionError(f"Rate limit exceeded (시도 {attempt_count}회)")
    return llm.invoke("코드를 분석해주세요").content

attempt_count = 0
result = retry_with_backoff(
    fn=flaky_llm_call,
    max_retries=3,
    base_delay=0.01,
    jitter=False,
    retriable_exceptions=(ConnectionError,),
)

assert attempt_count == 3
assert "분석" in result
print(f"FakeLLM 연동 테스트 통과: {attempt_count}번 시도 후 성공")
print(f"응답: {result}")
print("실습 3 완료! 모든 실습 통과")